In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *

spark = SparkSession.builder.appName('prac2').getOrCreate()

df = spark.read.csv('/Volumes/mycatalog/myschema/myvolume/employees_12000_with_nulls.csv', header = True, inferSchema = True)


**1. Find the top-performing employee in each department based on performance_score.**

In [0]:
df.withColumn('rank', dense_rank().over(Window.partitionBy('department').orderBy(desc('performance_score')))).where('rank = 1').display()



**2. Compute a running total of salary within each department, ordered by joining_date.**

In [0]:
df.withColumn('running_sal', sum('salary').over(Window.partitionBy('department').orderBy('joining_date'))).display()

**3. List employees whose salary is above the average salary of their department.**

In [0]:
dept_avg_sal = df.groupBy('department').agg(round(avg('salary'), 2).alias('avg')).select(coalesce(col('department'), lit('Temp')).alias('department'), 'avg')

new_df = df.select('id', 'name', coalesce('department', lit('Temp')).alias('department'), 'salary')

new_df.alias('a').join(dept_avg_sal.alias('b'), col('a.department') == col('b.department')).select('a.id', 'a.name', 'a.department', 'a.salary', 'b.avg').where('salary > avg').display()


**4. Identify the employee(s) with the earliest joining_date in each department.**

In [0]:
df.withColumn('rank', dense_rank().over(Window.partitionBy('department').orderBy('joining_date'))).where('rank = 1').display()

**5. Calculate the male-to-female ratio in each department, excluding nulls.**

In [0]:
dept_male = df.where('gender = "Male"').groupBy('department').agg(count('*').alias('Male_count'))
dept_female = df.where('gender = "Female"').groupBy('department').agg(count('*').alias('Female_count'))

dept_male.alias('a').join(dept_female.alias('b'), col('a.department') == col('b.department')).select('a.department', 'male_count', 'female_count', round(expr('male_count/female_count'), 2).alias('ratio')).display()

**6. Assign percentile rank to employees by salary within their department.**

In [0]:
df.select('id', 'salary', 'department', percent_rank().over(Window.partitionBy('department').orderBy(coalesce('salary', lit(0)))).alias('percent_rank')).display()

**7. Categorize employees into performance bands (High, Medium, Low) and count how many fall into each.**

In [0]:
df.withColumn('performance_band', when(col('performance_score') < 2, 'low')
                                  .when((col('performance_score') >= 2) & (col('performance_score') < 4), 'medium')
                                  .otherwise('high')).groupBy('performance_band').count().display()

8. Count the number of null values in each column.

In [0]:
id_col = df.where('id is NULL').select(expr('"ID" as column'), count('*').alias('null_count'))
name_col = df.where('name is NULL').select(expr('"name" as column'), count('*').alias('null_count'))
age_col = df.where('age is NULL').select(expr('"age" as column'), count('*').alias('null_count'))
gender_col = df.where('gender is NULL').select(expr('"gender" as column'), count('*').alias('null_count'))
department_col = df.where('department is NULL').select(expr('"department" as column'), count('*').alias('null_count'))
salary_col = df.where('salary is NULL').select(expr('"salary" as column'), count('*').alias('null_count'))
joining_date_col = df.where('joining_date is NULL').select(expr('"joining_date" as column'), count('*').alias('null_count'))
performance_score_col = df.where('performance_score is NULL').select(expr('"performance_score" as column'), count('*').alias('null_count'))

id_col.union(name_col).union(age_col).union(gender_col).union(department_col).union(salary_col).union(joining_date_col).union(performance_score_col).display()

**9. Group employees into age buckets (<30, 30–40, 40–50, >50) and calculate average salary per group.**

In [0]:
df.withColumn('age_group', when((col('age') < 30), '<30')
                          .when((col('age') >= 30) & (col('age') < 40), '30-40')
                          .when((col('age') >= 40) & (col('age') < 50), '40-50')
                          .otherwise('>50')).groupBy('age_group').agg(round(avg('salary'), 2).alias('avg_salary')).display()

**10. List employees who joined after 2020 and have a performance_score above 4.0.**

In [0]:
# df.where((year('joining_date') >= lit(2020)) & (col('performance_score') > lit(4))).display()

df.where('year(joining_date) >= 2020 and performance_score > 4').display()


**11. Identify the department with the highest average performance_score.**

In [0]:
df.groupBy(coalesce('department', lit('None'))).agg(round(avg('performance_score'), 2).alias('avg')).orderBy(desc('avg')).limit(1).display()

**12. Count how many employees were hired in each year-month (YYYY-MM).**

In [0]:
df.withColumn('year', year('joining_date')).withColumn('month', month('joining_date')).groupBy('year', 'month').count().orderBy('year', 'month').display()

**13. Retrieve the top 3 highest-paid employees in each department.**

In [0]:
df.withColumn('rank', dense_rank().over(Window.partitionBy('department').orderBy(desc('salary')))).where('rank <= 3').display()

**14. Show each employee’s performance_score and its deviation from the department’s average.**

**15. List employees whose salary is higher than any employee in the HR department.**

In [0]:
# SELECT *
# FROM table
# WHERE department != 'HR' AND salary > ANY (SELECT salary FROM table WHERE department = 'HR')

# SELECT *
# FROM table
# WHERE department != 'HR' AND salary > (SELECT min(salary) FROM table WHERE department = 'HR')

min_hr_sal = df.where('department = "HR"').select(min('salary').alias('min'))

df.join(min_hr_sal).where('salary > min').display()

**16. Find employees who have at least one null value in age, gender, department, salary, or performance_score.**

In [0]:
df.where('age is null or gender is null or department is null or salary is null or performance_score is null').display()

**17. Estimate current salary by applying 5% annual growth from their joining year.**

In [0]:
df.withColumn('years', expr('timestampdiff(year, joining_date, curdate())')).withColumn('new_sal', round(expr('salary * power((1.05), years)'), 0)).select('id', 'name', 'salary', 'new_sal').display()

**18. Find duplicate names appearing more than once in the dataset.**

In [0]:
df.groupby('name').agg(count('*').alias('count')).where('count > 1').display()

**19. Simulate reassigning all HR employees to Engineering and compute the new average salary in Engineering.**

In [0]:
df.where('department in ("HR", "Engineering")').select(round(avg('salary'), 0)).display()

**20. Calculate the total cost to company (CTC) per department, assuming CTC = salary + 30% of salary.**

In [0]:
df.withColumn('CTC', round(expr('salary * 1.3'), 0)).groupBy('department').agg(sum('CTC').alias('Total_CTC')).display()